A Rust router with a thin Python launcher: it spreads OpenAI-style requests over worker servers and health-checks them. Example 4 runs it for real, entirely on the loopback address: a toy worker in a thread, the `vllm-router` program in front of it, one request through. No outside network, no GPU, no model.

## Example 1 — the installed version, and the Rust extension really loaded

In [ ]:
import importlib.metadata
import vllm_router
import vllm_router_rs

assert vllm_router.__version__ == importlib.metadata.version("vllm_router"), vllm_router.__version__
# `vllm_router/__init__.py` hides a failed import of the Rust extension (it then exports only `__version__`),
# so the export list is what proves the extension is there.
assert vllm_router.__all__ == ["__version__", "Router"], vllm_router.__all__
policies = sorted(name for name in dir(vllm_router_rs.PolicyType) if not name.startswith("_"))
assert policies == ["CacheAware", "ConsistentHash", "PowerOfTwo", "Random", "RoundRobin"], policies
print("vllm_router", vllm_router.__version__, "with policies", policies)

## Example 2 — command-line arguments become router settings

In [ ]:
from vllm_router.launch_router import parse_router_args
from vllm_router.router import policy_from_str
from vllm_router_rs import PolicyType

defaults = parse_router_args([])
assert (defaults.worker_urls, defaults.policy, defaults.host, defaults.port) == ([], "cache_aware", "127.0.0.1", 30000)
chosen = parse_router_args(["--worker-urls", "http://127.0.0.1:8001", "http://127.0.0.1:8002",
                            "--policy", "round_robin", "--port", "3999"])
assert chosen.worker_urls == ["http://127.0.0.1:8001", "http://127.0.0.1:8002"] and chosen.port == 3999
assert policy_from_str(chosen.policy) == PolicyType.RoundRobin and policy_from_str(None) is None
split = parse_router_args(["--vllm-pd-disaggregation", "--prefill", "http://127.0.0.1:8001", "9000",
                           "--decode", "http://127.0.0.1:8002"])
assert split.prefill_urls == [("http://127.0.0.1:8001", 9000)] and split.decode_urls == ["http://127.0.0.1:8002"]
print("defaults: policy", defaults.policy, "port", defaults.port, "; two workers parsed; prefill/decode split parsed")

## Example 3 — the Rust router object is built from those settings (not started)

In [ ]:
from vllm_router.launch_router import parse_router_args
from vllm_router.router import Router
import vllm_router_rs

settings = parse_router_args(["--worker-urls", "http://127.0.0.1:8001", "--policy", "random"])
settings._validate_router_args()
router = Router.from_args(settings)
assert isinstance(router._router, vllm_router_rs.Router) and callable(router.start)
print("built the Rust router object", type(router._router).__name__, "for", settings.worker_urls)

## Example 4 — a request really goes through the router, on the loopback address only

In [ ]:
import http.server, json, shutil, socket, subprocess, threading, time, urllib.request

class Worker(http.server.BaseHTTPRequestHandler):          # a toy worker: answers every GET with the path it was asked
    def do_GET(self):
        body = json.dumps({"served": self.path}).encode()
        self.send_response(200), self.send_header("Content-Length", str(len(body))), self.end_headers()
        self.wfile.write(body)
    log_message = lambda self, *arguments: None            # keep the worker's request log out of the cell output
worker = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Worker)
threading.Thread(target=worker.serve_forever, daemon=True).start()
with socket.socket() as probe:                             # a free loopback port for the router
    probe.bind(("127.0.0.1", 0)); port = probe.getsockname()[1]
command = [shutil.which("vllm-router"), "--host", "127.0.0.1", "--port", str(port), "--worker-startup-check-interval", "1",
           "--worker-urls", f"http://127.0.0.1:{worker.server_address[1]}"]
router = subprocess.Popen(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
answer, deadline = None, time.monotonic() + 60
try:
    while answer is None and time.monotonic() < deadline and router.poll() is None:
        try:
            answer = json.load(urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=2))
        except OSError:                                    # the router is not listening yet
            time.sleep(0.3)
finally:
    router.terminate(), router.wait(timeout=30), worker.shutdown()
assert answer == {"served": "/v1/models"}, (answer, router.returncode)
print("the router came up, found its worker healthy and proxied GET /v1/models:", answer)

## Example 5 — settings that cannot work are refused before anything starts

In [ ]:
from vllm_router.launch_router import parse_router_args

try:
    parse_router_args(["--vllm-pd-disaggregation"])._validate_router_args()
    raise AssertionError("prefill/decode mode without a prefill server was accepted")
except ValueError as error:
    print("refused:", error)
try:
    parse_router_args(["--policy", "fastest"])
    raise AssertionError("an unknown policy was accepted")
except SystemExit as stopped:
    assert stopped.code == 2, stopped.code
    print("refused: unknown policy (argparse exit status 2)")